# Vocabulary Registry
> Typed, query-able vocabulary registry for CogitareLink.

Loads `data/registry_data.json` at import-time, validates each
entry with `pydantic`, and exposes convenient search helpers.

In [ ]:
#| default_exp vocab.registry

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import hashlib, json, importlib.resources as pkg
from datetime import datetime
from functools import lru_cache
from typing import Any, Dict, List, Union

import httpx
from pydantic import BaseModel, Field, AnyUrl, model_validator
from urllib.parse import urlparse, urlunparse

from cogitarelink.core.debug import get_logger
from cogitarelink.core.cache import InMemoryCache

## 1. Logger and optional RDFLib

In [ ]:
#| export
log = get_logger("registry")

# optional rdflib backend
try:
    from rdflib import Graph
    _HAS_RDFLIB = True
except ModuleNotFoundError:
    _HAS_RDFLIB = False

In [ ]:
#| hide

# Load bundled JSON once
_RAW_JSON: Dict[str, Any] = json.loads(
    pkg.files("cogitarelink").joinpath("data/registry_data.json").read_text()
)

## 2 - tiny HTTP helper

In [ ]:
#| export
_cache = InMemoryCache(maxsize=256)          # shared cache instance

@_cache.memoize("http")
def _http_get(url: str) -> bytes:
    "10 s-timeout HTTP GET with namespace-scoped cache."
    log.debug(f"GET {url}")
    r = httpx.get(url, follow_redirects=True, timeout=10)
    r.raise_for_status()
    return r.content

## 3 - Pydantic model blocks

In [ ]:
#| export
class ContextBlock(BaseModel):
    "Exactly **one** of `url`, `inline`, `derives_from` must be provided."
    url:          AnyUrl | None = None        # remote .jsonld
    inline:       Dict[str, Any] | None = None
    derives_from: AnyUrl | None = None        # .ttl, .rdf, â€¦

    sha256: str | None = None                # filled on first fetch

    @model_validator(mode="after")
    def _single_source(cls, v):
        if sum(x is not None for x in (v.url, v.inline, v.derives_from)) != 1:
            raise ValueError("Provide exactly one of url / inline / derives_from")
        return v

In [ ]:
#| export
class Versions(BaseModel):
    current:   str
    supported: List[str] = Field(default_factory=list)

### VocabEntry

In [ ]:
#| export
class VocabEntry(BaseModel):
    prefix:    str
    uris:      Dict[str, Union[AnyUrl, List[AnyUrl]]]  # {"primary": .., "alternates":[..]}
    context:   ContextBlock
    versions:  Versions

    features:  set[str] = Field(default_factory=set)
    tags:      set[str] = Field(default_factory=set)
    strategy_defaults: Dict[str, str] = Field(default_factory=dict)
    meta:      Dict[str, Any] = Field(default_factory=dict)

    # ------------------------------------------------------------------ #
    # public API
    # ------------------------------------------------------------------ #
    def context_payload(self) -> Dict[str, Any]:
        "Return (and cache) the merged JSON-LD @context dict."
        return _load_ctx(self.prefix, self.versions.current)   # see below

## 4 - context loader (private)

In [ ]:
#| export
@lru_cache(maxsize=256)
def _load_ctx(prefix: str, version: str) -> Dict[str, Any]:
    """Internal LRU-cached loader for a given prefix+version."""
    e = registry[prefix]

    # pick raw bytes -----------------------------------------------------
    if e.context.inline is not None:                     # already a dict
        raw_dict = e.context.inline
    elif e.context.url is not None:                      # remote .jsonld
        raw_dict = json.loads(_http_get(str(e.context.url)))
    else:                                                # derives_from *.ttl
        if not _HAS_RDFLIB:
            raise RuntimeError("Deriving context requires `rdflib` installed")
        ttl = _http_get(str(e.context.derives_from))
        g = Graph().parse(data=ttl, format="turtle")
        raw_dict = {"@context": {p: str(iri) for p, iri in g.namespaces()}}

    # compute sha once and persist back into in-memory entry -------------
    s = hashlib.sha256(
        json.dumps(raw_dict, sort_keys=True, separators=(",", ":")).encode()
    ).hexdigest()
    e.context.sha256 = s

    return raw_dict

## 5- Registry Singleton

In [ ]:
#| export
class _Registry:
    "Read-only registry; supports prefix *and* alias URL look-ups."

    def __init__(self):
        fp = pkg.files("cogitarelink").joinpath("data/registry_data.json")
        raw: Dict[str, Any] = json.loads(fp.read_text())

        self._v: Dict[str, VocabEntry] = {k: VocabEntry(**v) for k, v in raw.items()}

        # build alias map â†’ prefix
        self._alias: Dict[str, str] = {}
        for p, e in self._v.items():
            for url in e.uris.values():
                self._alias[self._norm(url)] = p

    # ---------------- basic mapping protocol --------------------------
    def __getitem__(self, p: str) -> VocabEntry:
        return self._v[p]

    def __iter__(self):
        return iter(self._v.values())

    # ---------------- convenience helpers ----------------------------
    def resolve(self, ident: str) -> VocabEntry:
        "Accept prefix **or** any alias URI."
        if ident in self._v:
            return self._v[ident]
        try:
            return self._v[self._alias[self._norm(ident)]]
        except KeyError as e:
            raise KeyError(f"{ident!r} not found in registry") from e

    @staticmethod
    def _norm(u: str) -> str:
        p = urlparse(str(u))
        return urlunparse((p.scheme.lower(), p.netloc.lower(), p.path.rstrip("/"),
                           "", "", ""))


registry = _Registry()

## 6 - Collision Hint Helper

In [ ]:
#| export
def preferred_collision(a: str, b: str) -> Dict[str, str] | None:
    """Return strategy hint if vocab *a* nominates one for *b*."""
    try:
        return registry[a].strategy_defaults.get(b)     # type: ignore
    except KeyError:
        return None

In [ ]:
#| hide
from fastcore.test import *

# prefix vs URL resolution
schema = registry["schema"]
test_eq(schema, registry.resolve("https://schema.org/"))

# Turtle derivation stays deterministic
if schema.context.derives_from:
    c1 = schema.context_payload()
    c2 = schema.context_payload()
    test_eq(c1, c2)

# exclusivity validation
with ExceptionExpected(ValueError):
    ContextBlock()                                      # none provided

In [ ]:

ctx = registry["vc"].context_payload()          # fetch + cache
print(ctx["@context"]["type"])                  # proves we got JSON-LD

@type


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()